# Load the GPT-2 Small weights and generate text

In Chapter 3 we built `GPTModel`, a GPT-2-like model, but its weights were
random, so its output was meaningless. Training a model of this size from
scratch needs a large dataset and a lot of compute. Fortunately, OpenAI
published the weights of GPT-2: in this notebook we load the weights of
**GPT-2 Small** (124M parameters) into our own model, and use it to continue a
text.

The steps:

1. download the weights from Hugging Face;
2. read the checkpoint and look at its tensors;
3. create a `GPTModel` with the GPT-2 Small configuration;
4. understand how the GPT-2 tensors differ from the parameters of our model;
5. convert the tensors and load them into the model;
6. prepare the model for inference;
7. tokenize a prompt;
8. generate a continuation with `generate_text_simple`.

The model code comes from [`build_llm/gpt2.py`](../build_llm/gpt2.py); the
download and the conversion of the weights from
[`build_llm/gpt2_weights.py`](../build_llm/gpt2_weights.py).

## A different path from the book

The book loads the weights in a different way. OpenAI released the original
GPT-2 weights as **TensorFlow checkpoints**, and the book downloads those
files and reads them with TensorFlow.

This repository does **not** use TensorFlow. It is a large framework, and we
would install it only to read a few files once: everything else here is
PyTorch.

The same weights are also published by Hugging Face, in the repository
[`openai-community/gpt2`](https://huggingface.co/openai-community/gpt2), in
the **safetensors** format:

* it is a simple format: a small JSON header with the name, data type, and
  shape of every tensor, followed by the raw numbers;
* it is **safe**: reading a file never executes code, unlike the pickle-based
  `.bin` and `.pth` files of PyTorch;
* it is read by the small `safetensors` library, directly into PyTorch
  tensors, without TensorFlow or the `transformers` library.

The numbers are the same, so the final model is the same as in the book. What
changes is how we read the file, and the names and layout of the tensors, which
follow the Hugging Face implementation of GPT-2. Most of this notebook is
about that conversion.

In [1]:
import tiktoken
import torch

from build_llm.device import get_device
from build_llm.gpt2 import GPTModel, generate_text_simple
from build_llm.gpt2_weights import (
    GPT2_SMALL_CONFIG,
    convert_hf_gpt2_state_dict,
    download_gpt2_small_weights,
    load_hf_gpt2_weights,
    load_hf_state_dict,
)
from build_llm.model_parameters import count_parameters

device = get_device()
print(f"Device: {device}")

Device: mps


## Step 1: download the weights

`download_gpt2_small_weights` downloads `model.safetensors` (about 548 MB)
into `models/gpt2-small/`, in the repository root. The folder is ignored by
Git. The details:

* the URL points to a fixed revision of the Hugging Face repository, so the
  file never changes;
* after the download, the SHA-256 checksum of the file is compared with the
  one published by Hugging Face;
* if a valid file is already there, nothing is downloaded: only the first run
  needs the network.

In [2]:
weights_path = download_gpt2_small_weights()
print(weights_path.name, f"{weights_path.stat().st_size / 1e6:.0f} MB")

model.safetensors 548 MB


## Step 2: read the checkpoint

A checkpoint is a **state dict**: a dictionary that maps the name of each
tensor to its values. `load_hf_state_dict` reads the safetensors file into
such a dictionary, with all the tensors on the CPU.

Let's look at the names and the shapes.

In [3]:
hf_state_dict = load_hf_state_dict(weights_path)
print(f"Tensors in the checkpoint: {len(hf_state_dict)}")
print()

print("Outside the transformer blocks:")
for name in ["wte.weight", "wpe.weight", "ln_f.weight", "ln_f.bias"]:
    print(f"  {name:<28} {tuple(hf_state_dict[name].shape)}")

print()
print("The first transformer block, h.0:")
block_0 = sorted(name for name in hf_state_dict if name.startswith("h.0."))
for name in block_0:
    print(f"  {name:<28} {tuple(hf_state_dict[name].shape)}")

Tensors in the checkpoint: 160

Outside the transformer blocks:
  wte.weight                   (50257, 768)
  wpe.weight                   (1024, 768)
  ln_f.weight                  (768,)
  ln_f.bias                    (768,)

The first transformer block, h.0:
  h.0.attn.bias                (1, 1, 1024, 1024)
  h.0.attn.c_attn.bias         (2304,)
  h.0.attn.c_attn.weight       (768, 2304)
  h.0.attn.c_proj.bias         (768,)
  h.0.attn.c_proj.weight       (768, 768)
  h.0.ln_1.bias                (768,)
  h.0.ln_1.weight              (768,)
  h.0.ln_2.bias                (768,)
  h.0.ln_2.weight              (768,)
  h.0.mlp.c_fc.bias            (3072,)
  h.0.mlp.c_fc.weight          (768, 3072)
  h.0.mlp.c_proj.bias          (768,)
  h.0.mlp.c_proj.weight        (3072, 768)


The names come from the Hugging Face implementation of GPT-2, which keeps
the names of the original OpenAI code:

| GPT-2 name | Meaning | In our `GPTModel` |
| --- | --- | --- |
| `wte` | word token embeddings, `(50257, 768)` | `tok_emb` |
| `wpe` | word position embeddings, `(1024, 768)` | `pos_emb` |
| `h.0` ... `h.11` | the 12 transformer blocks | `trf_blocks.0` ... `trf_blocks.11` |
| `ln_1`, `ln_2` | the two layer normalizations of a block | `norm1`, `norm2` |
| `weight`, `bias` of a layer norm | scale and shift | `scale`, `shift` |
| `attn.c_attn` | query, key, and value projections, **fused** | `att.W_query`, `att.W_key`, `att.W_value` |
| `attn.c_proj` | output projection of attention | `att.out_proj` |
| `mlp.c_fc` | first feed-forward layer, 768 to 3072 | `ff.layers.0` |
| `mlp.c_proj` | second feed-forward layer, 3072 to 768 | `ff.layers.2` |
| `ln_f` | final layer normalization | `final_norm` |

Two more observations:

* `h.N.attn.bias`, of shape `(1, 1, 1024, 1024)`, is **not a trained
  parameter**: it is the causal mask, saved with the model. Our model computes
  its own mask, so we ignore it.
* There is **no output head** in the checkpoint. GPT-2 uses **weight
  tying**: the output head reuses the token embedding matrix `wte`, which has
  the right shape, `(vocab_size, emb_dim)`.

## Step 3: create the model

`GPT2_SMALL_CONFIG` has the dimensions of GPT-2 Small. Compared with the
configuration of Chapter 3, one value changes: `qkv_bias` is `True`, because
GPT-2 has biases in the query, key, and value projections. Without them, the
model would have no place for the `c_attn.bias` tensors.

The dropout rate does not matter here: we will switch the model to evaluation
mode, where dropout is disabled.

In [4]:
print(GPT2_SMALL_CONFIG)

torch.manual_seed(123)
model = GPTModel(GPT2_SMALL_CONFIG)
print(f"Parameters of GPTModel: {count_parameters(model):,}")

{'vocab_size': 50257, 'context_length': 1024, 'emb_dim': 768, 'n_heads': 12, 'n_layers': 12, 'drop_rate': 0.1, 'qkv_bias': True}


Parameters of GPTModel: 163,037,184


The model has about 163M parameters, not 124M: our output head has its own
`(50257, 768)` matrix, while GPT-2 shares it with the token embeddings. We
will fill the output head with a **copy** of `wte`. The result is the same as
in GPT-2, at the cost of about 38.6M duplicated values (about 150 MB in
float32).

For now, the weights are random. Let's see what this model writes, to compare
it later with the trained one. First, the tokenizer: GPT-2 uses the `gpt2` BPE
encoding of tiktoken, the one we explored in Chapter 1.

In [5]:
tokenizer = tiktoken.get_encoding("gpt2")


def text_to_ids(text):
    """Encode a text as a batch with one sequence: (1, num_tokens)."""
    return torch.tensor([tokenizer.encode(text)])


def ids_to_text(token_ids):
    """Decode the first sequence of a batch of token IDs."""
    return tokenizer.decode(token_ids[0].tolist())


prompt = "Every effort moves you"
model.eval()
random_output = generate_text_simple(
    model, text_to_ids(prompt), max_new_tokens=10, context_size=1024
)
print(ids_to_text(random_output))

Every effort moves youarcity warrQu motivateYS Coreints kernel 220redited


With random weights, the model picks unrelated tokens: the text has no
meaning.

## Step 4: three differences between GPT-2 and our model

Before loading, we must solve three differences between the checkpoint and
our model.

**1. The names.** They follow the table above: for example, `h.0.ln_1.weight`
becomes `trf_blocks.0.norm1.scale`.

**2. The fused query, key, and value projections.** GPT-2 stores the three
projections in one matrix, `c_attn`, of shape `(768, 2304)`: the first 768
columns are the query projection, the next 768 the key projection, and the
last 768 the value projection. We split it into three parts.

In [6]:
c_attn_weight = hf_state_dict["h.0.attn.c_attn.weight"]
print(f"c_attn.weight: {tuple(c_attn_weight.shape)}")

query_part, key_part, value_part = c_attn_weight.chunk(3, dim=1)
print(f"Each part:     {tuple(query_part.shape)}")

our_query = model.trf_blocks[0].att.W_query.weight
print(f"W_query.weight in our model: {tuple(our_query.shape)}")

c_attn.weight: (768, 2304)
Each part:     (768, 768)
W_query.weight in our model: (768, 768)


**3. The layout of the weights.** Each part is `(768, 768)`, like
`W_query.weight`, but the shapes are not enough: the matrices are
**transposed** with respect to each other.

GPT-2 uses a layer called `Conv1D`, which, despite its name, is a linear layer
that stores its weight as `(in_features, out_features)` and computes:

```text
y = x @ W + b
```

`nn.Linear` stores its weight as `(out_features, in_features)` and computes
`y = x @ W.T + b`. So we must transpose every GPT-2 projection: `W.T`. For a
square `(768, 768)` matrix, forgetting the transposition would raise no error,
but the model would compute wrong results.

The feed-forward layers show the difference more clearly, because they are
not square:

In [7]:
c_fc_weight = hf_state_dict["h.0.mlp.c_fc.weight"]
c_fc_bias = hf_state_dict["h.0.mlp.c_fc.bias"]
print(f"GPT-2 mlp.c_fc.weight:   {tuple(c_fc_weight.shape)}  (in, out)")
print(
    f"Our ff.layers.0.weight:  {tuple(model.trf_blocks[0].ff.layers[0].weight.shape)}  (out, in)"
)

# The two conventions compute the same function, once W is transposed.
x = torch.randn(2, 768)
conv1d_output = x @ c_fc_weight + c_fc_bias
linear_output = torch.nn.functional.linear(x, c_fc_weight.T, c_fc_bias)
print("Same result:", torch.allclose(conv1d_output, linear_output, atol=1e-5))

GPT-2 mlp.c_fc.weight:   (768, 3072)  (in, out)
Our ff.layers.0.weight:  (3072, 768)  (out, in)
Same result: True


## Step 5: convert and load the weights

`convert_hf_gpt2_state_dict` applies the three conversions to every tensor:
it renames them, splits `c_attn`, transposes the projection weights, and uses
`wte` also for the output head. The result is a state dict with the names of
our model.

In [8]:
converted = convert_hf_gpt2_state_dict(hf_state_dict, GPT2_SMALL_CONFIG["n_layers"])
print(f"Converted tensors: {len(converted)}")

our_names = set(model.state_dict())
not_converted = sorted(our_names - set(converted))
print(f"Model entries not in the converted dict: {len(not_converted)}")
print("For example:", not_converted[:2])

Converted tensors: 197
Model entries not in the converted dict: 12
For example: ['trf_blocks.0.att.mask', 'trf_blocks.1.att.mask']


Every parameter of the model is covered. The only entries left are the 12
causal masks, which are buffers computed by the model itself, not trained
weights.

`load_hf_gpt2_weights` performs the conversion and calls PyTorch's
`load_state_dict`, which copies each tensor into the matching parameter. It
also checks that every shape matches: a wrong conversion would stop here with
an error, rather than produce a broken model silently. Let's load the weights
and check a few of them.

In [9]:
load_hf_gpt2_weights(model, hf_state_dict)

print(
    "Token embeddings equal wte:",
    torch.equal(model.tok_emb.weight, hf_state_dict["wte.weight"]),
)
print(
    "Output head equals wte:",
    torch.equal(model.out_head.weight, hf_state_dict["wte.weight"]),
)
print(
    "W_key of block 0 is the transposed key part:",
    torch.equal(model.trf_blocks[0].att.W_key.weight, key_part.T),
)

Token embeddings equal wte: True
Output head equals wte: True
W_key of block 0 is the transposed key part: True


## Step 6: prepare the model for inference

Two things before generating:

* **Move the model to the device**: the Apple Silicon GPU (MPS) when available,
  otherwise CUDA or the CPU. The inputs must be on the same device.
* **Switch to evaluation mode** with `model.eval()`. In training mode, dropout
  randomly zeroes 10% of the activations, so the output would change at every
  run. `generate_text_simple` does not change the mode of the model: it is our
  job.

The checkpoint is no longer needed, so we release its memory.

In [10]:
del hf_state_dict, converted

model.to(device)
model.eval()
print(f"Model on {next(model.parameters()).device}, training mode: {model.training}")

Model on mps:0, training mode: False


## Step 7: tokenize the prompt

The model reads token IDs, not text. `text_to_ids` encodes the prompt with the
GPT-2 tokenizer and adds the batch dimension: the model expects a tensor of
shape `(batch_size, num_tokens)`.

In [11]:
prompt_ids = text_to_ids(prompt).to(device)
print(f"Prompt: {prompt!r}")
print(f"Token IDs: {prompt_ids.tolist()}, shape {tuple(prompt_ids.shape)}")
print("Tokens:", [tokenizer.decode([token_id]) for token_id in prompt_ids[0].tolist()])

Prompt: 'Every effort moves you'
Token IDs: [[6109, 3626, 6100, 345]], shape (1, 4)
Tokens: ['Every', ' effort', ' moves', ' you']


## Step 8: generate the continuation

`generate_text_simple` repeats a simple loop, once per new token:

1. keep the last `context_size` tokens: the model cannot read more than its
   context length (1024 for GPT-2);
2. run the model, without gradients, and take the logits of the **last**
   position, which predict the next token: shape `(batch_size, vocab_size)`;
3. choose the token with the **largest logit** (greedy decoding);
4. append it to the sequence, and repeat.

The model never sees the text: only token IDs in, logits out. We decode the
result at the end.

In [12]:
output_ids = generate_text_simple(
    model,
    prompt_ids,
    max_new_tokens=25,
    context_size=GPT2_SMALL_CONFIG["context_length"],
)
print(f"Output shape: {tuple(output_ids.shape)}")
print()
print(ids_to_text(output_ids))

Output shape: (1, 29)

Every effort moves you forward.

The first step is to understand the importance of your work.

The second step is to understand the


Now the continuation is fluent English, while with random weights it was
nonsense. The architecture is the same: only the weights changed.

The output is the same every time: greedy decoding always picks the most
likely token, so there is no randomness. This also has a drawback, visible
with another prompt: the model can fall into a loop and repeat itself.

In [13]:
prompt = "Hello, I'm a language model,"
output_ids = generate_text_simple(
    model,
    text_to_ids(prompt).to(device),
    max_new_tokens=25,
    context_size=GPT2_SMALL_CONFIG["context_length"],
)
print(ids_to_text(output_ids))

Hello, I'm a language model, not a programming language. I'm a language model. I'm a language model. I'm a language model. I'm


Repetition is a well-known weakness of greedy decoding. Generation
strategies such as **temperature** and **top-k sampling** pick the next token
at random among the likely ones, and produce more varied text. They are the
next step.

## Summary

* The GPT-2 weights are published by OpenAI in TensorFlow format, and by
  Hugging Face also in the safetensors format, which we read without
  TensorFlow.
* A checkpoint is a state dict: names mapped to tensors. Its names follow the
  GPT-2 code, so they must be mapped to the names of our model.
* GPT-2 fuses the query, key, and value projections into `c_attn`, and its
  `Conv1D` layers store transposed weights: we split and transpose them.
* GPT-2 ties the output head to the token embeddings, so we copy `wte` into
  `out_head`.
* After loading, the same `GPTModel` of Chapter 3 writes fluent text.

To free the disk space, delete the folder `models/gpt2-small/` in the
repository root: the next run downloads the file again.